In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

# 1. Load the engineered data from Week 2
df = pd.read_csv('../../data/engineered_earthquake_data.csv')

# 2. Recreate our balanced Target Variable (High Risk vs Low Risk)
threshold = df['magnitude'].median()
df['High_Risk'] = (df['magnitude'] >= threshold).astype(int)

# 3. Define Features and Target
features = ['depth_scaled', 'latitude', 'longitude', 'location_risk_score']
X = df[features]
y = df['High_Risk']

# 4. Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Week 4 Setup Complete!")
print(f"Training data size: {X_train.shape[0]} rows")

Week 4 Setup Complete!
Training data size: 625 rows


In [2]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score

# 1. Initialize Random Forest
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)

# 2. Run 5-Fold Cross Validation
print("Running Cross-Validation... (This tests the model 5 times)")
cv_scores = cross_val_score(rf_model, X_train, y_train, cv=5)
print(f"Scores for each fold: {cv_scores}")
print(f"Average CV Accuracy: {cv_scores.mean() * 100:.2f}%\n")

# 3. Train on full training data and test
rf_model.fit(X_train, y_train)
rf_preds = rf_model.predict(X_test)
rf_accuracy = accuracy_score(y_test, rf_preds)

print(f"Standard Random Forest Test Accuracy: {rf_accuracy * 100:.2f}%")

Running Cross-Validation... (This tests the model 5 times)
Scores for each fold: [0.536 0.6   0.616 0.576 0.488]
Average CV Accuracy: 56.32%

Standard Random Forest Test Accuracy: 59.24%


In [3]:
from sklearn.ensemble import GradientBoostingClassifier

# 1. Initialize the Gradient Boosting model
gb_model = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=42)

# 2. Train the model
gb_model.fit(X_train, y_train)

# 3. Predict and evaluate
gb_preds = gb_model.predict(X_test)
gb_accuracy = accuracy_score(y_test, gb_preds)

print(f"Gradient Boosting Test Accuracy: {gb_accuracy * 100:.2f}%")

Gradient Boosting Test Accuracy: 52.23%


In [4]:
from sklearn.model_selection import GridSearchCV

# 1. Define the settings we want the computer to test
# n_estimators = number of trees, max_depth = how deep the trees can grow
param_grid = {
    'n_estimators': [50, 100, 150], 
    'max_depth': [10, 20, None]
}

print("Testing combinations to find the best settings... (Please wait)")

# 2. Set up Grid Search on our Random Forest
grid_search = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, cv=3)

# 3. Run the search
grid_search.fit(X_train, y_train)

# 4. Extract the absolute best model
best_rf = grid_search.best_estimator_
tuned_preds = best_rf.predict(X_test)
tuned_accuracy = accuracy_score(y_test, tuned_preds)

print(f"\nBest Settings Found: {grid_search.best_params_}")
print(f"Tuned Random Forest Accuracy: {tuned_accuracy * 100:.2f}%")

Testing combinations to find the best settings... (Please wait)

Best Settings Found: {'max_depth': 20, 'n_estimators': 50}
Tuned Random Forest Accuracy: 56.05%
